# Check Data

In [1]:
import pandas as pd
import math
from scipy.stats import chisquare, chi2_contingency
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep

df = pd.read_csv("../data/raw/marketing_AB.csv")

In [2]:
# Drop the unecessary index column
df = df.drop(columns='Unnamed: 0')

In [3]:
df.head(10)

,user id,test group,converted,total ads,most ads day,most ads hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14
5,1137664,ad,False,734,Saturday,10
6,1116205,ad,False,264,Wednesday,13
7,1496843,ad,False,17,Sunday,18
8,1448851,ad,False,21,Tuesday,19
9,1446284,ad,False,142,Monday,14


In [4]:
# Make columns into snake case
df.columns = df.columns.str.replace(' ', '_')

In [5]:
# Checking shape, data types, nulls, and duplicates
print(df.shape)
print("\n")
print(df.dtypes)
print("\n")
print(df.isna().sum())
print("\n")
print(df.duplicated().sum())
print(df['user_id'].duplicated().sum())

(588101, 6)


user_id          int64
test_group         str
converted         bool
total_ads        int64
most_ads_day       str
most_ads_hour    int64
dtype: object


user_id          0
test_group       0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
dtype: int64


0
0


In [6]:
# Changing column converted to 0/1 boolean instead of true or false
df['converted'] = df['converted'].astype(int)

In [7]:
# Group sizes
print(df.groupby('test_group').size())
(df.groupby('test_group').size() / len(df)) * 100

test_group
ad     564577
psa     23524
dtype: int64


test_group
ad     96.000007
psa     3.999993
dtype: float64

In [8]:
summary = (
    df.groupby('test_group')['converted']
    .agg(total='count', conversions='sum', conversion_rate='mean')
    .reset_index()
)

summary['conversion_rate_pct'] = (summary['conversion_rate'] * 100).round(2)
summary

,test_group,total,conversions,conversion_rate,conversion_rate_pct
0,ad,564577,14423,0.025547,2.55
1,psa,23524,420,0.017854,1.79


# Sample Ratio Check

In [9]:
# The groups are very unbalanced (96% VS. 4%)
counts = df['test_group'].value_counts()
observed = [counts['ad'], counts['psa']]

total = sum(observed)
expected = [total * 0.96, total * 0.04]

chi2_stat, p_value = chisquare(f_obs = observed, f_exp=expected)

print(f"Observed [ad, psa]: {observed}")
print(f"Expected [ad, psa]: {expected}")
print(f"Chi2: {chi2_stat:.10f}, p-value: {p_value:.4f}")
# Chi square statistic is nearly 0 because the observed vs expected is nearly identical, p-value is well above
# any threshold to reject the null hypothesis (That there is no difference between the observed and expected groups).
# Which also means the small PSA group also limits the statistical power, having its variance contribution to be larger
# than the AD group's.

Observed [ad, psa]: [np.int64(564577), np.int64(23524)]
Expected [ad, psa]: [np.float64(564576.96), np.float64(23524.04)]
Chi2: 0.0000000708, p-value: 0.9998


# Hypothesis and Design

##### H0: Conversion rates are equal, Ha: Conversion rates are different
##### Will be using a two-sided test because:
1. Checks if ad performs differently from psa instead of strictly better or worse.
2. Prevents p-value from hitting tresholds easier, as one-sided tests cuts p-values in half.
3. More confidents when passing a two-sided test threshold.

In [10]:
alpha = 0.05
conversion_rate = df.groupby('test_group')['converted'].mean()
mde = 0.005
psa_control = conversion_rate['psa']
psa_treatment = conversion_rate['psa'] + mde

effect_size = proportion_effectsize(psa_treatment, psa_control)
power_analysis = NormalIndPower()

In [11]:
equal_split = power_analysis.solve_power(
    effect_size = effect_size,
    power = 0.80,
    alpha = alpha,
    ratio = 1.0,
    alternative ='two-sided'
)

actual_split = power_analysis.solve_power(
    effect_size = effect_size,
    power = 0.80,
    alpha = alpha,
    ratio = (564577 / 23524),
    alternative = 'two-sided'
)

In [12]:
n_trt_unequal = actual_split * (564577 / 23524)
print(f"50/50 Split -> Control: {math.ceil(equal_split):,}, Treatment: {math.ceil(equal_split):,} (Total: {math.ceil(equal_split)*2:,})")
print(f"96/4 Split  -> Control: {math.ceil(actual_split):,}, Treatment: {math.ceil(n_trt_unequal):,} (Total: {math.ceil(actual_split + n_trt_unequal):,})")
# Experiment requires 6,497 control (psa) and 155,921 (ad) users (162,418 total), where 50/50 split only required 24,948 total users.
# The observed dataset contains 23,524 control and 565,577 treatment users, which is approximately 3.62 times the 
# required sample size. Therefore, the experiment is adequately powered to reliably detect an effect of 0.5 percentage
# points or larger.

50/50 Split -> Control: 12,474, Treatment: 12,474 (Total: 24,948)
96/4 Split  -> Control: 6,497, Treatment: 155,921 (Total: 162,418)


# Main Significance Test
Testing whether the proportion of users who converted in the ad group is statistically different from the proportion of users who converted in the psa group

In [13]:
summary2 = df.groupby('test_group')['converted'].agg(['sum', 'count'])

ad_conv, ad_n = summary2.loc['ad', 'sum'], summary2.loc['ad', 'count']
psa_conv, psa_n = summary2.loc['psa', 'sum'], summary2.loc['psa', 'count']

ad_rate = ad_conv / ad_n
psa_rate = psa_conv / psa_n
abs_lift = ad_rate - psa_rate
rel_lift = (ad_rate - psa_rate) / psa_rate

In [14]:
z_stat, p_value = proportions_ztest(
    count = [ad_conv, psa_conv],
    nobs = [ad_n, psa_n],
    alternative = 'two-sided'
)

In [15]:
ci_low, ci_high = confint_proportions_2indep(
    count1 = ad_conv, nobs1 = ad_n,
    count2 = psa_conv, nobs2 = psa_n,
    method = 'wald'
)

In [29]:
print(f"Ad CR:          {ad_rate:.4%}")
print(f"PSA CR:         {psa_rate:.4%}")
print(f"Absolute Lift:  {abs_lift:+.4%} (95% CI: [{ci_low:+.4%}, {ci_high:+.4%}])")
print(f"Relative Lift:  {rel_lift:+.2%}")
print(f"Z-stat:         {z_stat:.4f}, p-value: {p_value:.4e}")

Ad CR:          2.5547%
PSA CR:         1.7854%
Absolute Lift:  +0.7692% (95% CI: [+0.5951%, +0.9434%])
Relative Lift:  +43.09%
Z-stat:         7.3701, p-value: 1.7053e-13


In [20]:
# Chi-square test for cross-checking
contingency_table = pd.crosstab(df['test_group'], df['converted'])
contingency_table

converted,0,1
test_group,,
ad,550154,14423
psa,23104,420


In [26]:
chi2_stat, p_value, df, expected = chi2_contingency(contingency_table, correction = False)

print(f"Chi2 Stat: {chi2_stat:.4f}")
print(f"P-Value: {p_value:.4e}")
print(f"Degrees of Freedom: {df}")
print(f"Expected Frequencies:\n {expected}")

Chi2 Stat: 54.3181
P-Value: 1.7053e-13
Degrees of Freedom: 1
Expected Frequencies:
 [[550327.71899045  14249.28100955]
 [ 22930.28100955    593.71899045]]


The results are statistically significance, users who saw the ad converted at a rate of 2.55%. compared to 1.79% of users who saw the public service announcement. Both two-proportion z-test and chi-square test of independence had a p-value below 0.0001, meaning there is virtually zero chance this difference occured by chance. Size of lift matters as well, the observed gain of +0.77 percentage points with a 95% confidence interval of [+0.5951%, +0.9434%], which beats the fixed Minimum Detectable Effect of 0.50 percentage points. 

Showing the ads drove a 43% relative increase in conversion rate, generating around 4,340 (14,423 - 10,080 expected baseline conversions/hypothetical) incremental conversions across the 564,577 users in the treatment group (ad) that would not have happened without the campaign.